## 分块

您的文档太大，无法作为一个整体嵌入。分块是在嵌入之前将它们分成更小的部分的过程

如何对文档进行分块直接影响系统查找相关信息并给出准确答案的能力，如果搜索准备不足的数据，即使完美的检索系统也会失败

**资源：**

1、 Weaviate：RAG 的分块策略（免费）

链接：<https://weaviate.io/blog/chunking-strategies-for-rag>

最实用的指南。涵盖固定大小、递归和语义分块，并提供关于何时使用每种分块的明确指导

2、 非结构化：RAG 最佳实践的分块（免费）

链接：<https://unstructed.io/blog/chunking-for-rag-best-practices>

对块大小、重叠以及嵌入模型的上下文窗口如何施加硬限制的技术深入探讨

实验的一个很好的起点是大约 250 个标记（大约 1,000 个字符）的块大小，并结合连续块之间 10-20% 的重叠，以避免在边界处丢失上下文

3、 LangChain 文本分割器文档（官方，免费）

链接：<https://docs.langchain.com/oss/javascript/integrations/splitters>

在代码中使用 RecursiveCharacterTextSplitter、MarkdownTextSplitter 和语义分割器的实用参考

**关注点：** 以重叠为基准的固定大小分块、结构化文档的递归分块、更好的边界检测的语义分块以及

**核心权衡：** 太大的块会失去检索精度；太小的块会丢失上下文

**初学者提示：** 从 LangChain 的 RecursiveCharacterTextSplitter 开始，其中 chunk_size=500 和 chunk_overlap=50。对于大多数文档来说，这是最明智的默认设置，并为您提供了一个改进的工作基准

### 基于文本结构的分块

文本自然地组织成段落、句子和单词等层次化单元。我们可以利用这种固有结构来指导我们的分割策略，创建保持自然语言流畅性、在分割内维持语义连贯性，并能适应不同文本粒度级别的分割。LangChain的RecursiveCharacterTextSplitter实现了这一概念：

- `RecursiveCharacterTextSplitter` 尝试保持较大单元（例如段落）的完整性。
- 如果一个单元超过了分块大小，它会移动到下一个层级（例如，句子）。
- 如有必要，此过程会一直向下进行到单词级别。

In [11]:
import { RecursiveCharacterTextSplitter } from '@langchain/textsplitters'

const splitter = new RecursiveCharacterTextSplitter({ chunkSize: 100, chunkOverlap: 0 })
const document = await Deno.readTextFile('../../data/qiu.txt')
console.log('文档长度:', document.length, '字符')
const texts = await splitter.splitText(document)
console.log('分割成', texts.length, '段文本')
console.log('第一段文本:\n', texts[0])


文档长度: 189496 字符
分割成 2625 段文本
第一段文本:
 三体前传：球状闪电 作者：刘慈欣


### 基于长度的分割

一种直观的策略是根据文档的长度进行分割。这种简单而有效的方法确保每个文本块不会超过指定的大小限制。基于长度分割的主要优势：

- 实现简单直接
- 文本块大小一致
- 易于适应不同的模型需求

基于长度的分割类型：

- 基于标记的分割：根据标记数量分割文本，这在处理语言模型时非常有用。
- 基于字符的分割：根据字符数量分割文本，这种方法在不同类型的文本中可能更加一致。

使用LangChain的 `CharacterTextSplitter` 结合基于标记的分割进行示例实现：

In [12]:
import { TokenTextSplitter } from '@langchain/textsplitters'

const splitter1 = new TokenTextSplitter({
  encodingName: 'cl100k_base',
  chunkSize: 100,
  chunkOverlap: 0,
})
const texts1 = await splitter1.splitText(document)
console.log('分割成', texts1.length, '段文本')
console.log('第一段文本:\n', texts1[0])


分割成 2299 段文本
第一段文本:
 三体前传：球状闪电 作者：刘慈欣

内容简介：
　　没有《球状闪电》，就没有后来的《三体》！
　　《三体》前传！
　　亚洲首位雨果奖得主刘慈欣的三大长篇之一！（《三体》《球状闪
第一段文本:
 三体前传：球状闪电 作者：刘慈欣

内容简介：
　　没有《球状闪电》，就没有后来的《三体》！
　　《三体》前传！
　　亚洲首位雨果奖得主刘慈欣的三大长篇之一！（《三体》《球状闪


### 基于文档结构

某些文档具有固有结构，例如HTML、Markdown或JSON文件。在这些情况下，基于文档结构进行分割是有益的，因为它通常自然地按语义对相关文本进行分组。基于结构分割的主要优点：

- 保留文档的逻辑组织
- 在每个块内保持上下文
- 对于下游任务（如检索或摘要）可能更有效

In [16]:
const TS_CODE = `
function helloWorld() {
  console.log("Hello, World!");
}

// Call the function
helloWorld();
`

const tsSplitter = RecursiveCharacterTextSplitter.fromLanguage(
  'js',
  { chunkSize: 60, chunkOverlap: 0 },
)
const tsDocs = await tsSplitter.createDocuments([TS_CODE])
console.log(tsDocs)


[
  Document {
    pageContent: 'function helloWorld() {\n  console.log("Hello, World!");\n}',
    metadata: { loc: { lines: { from: 2, to: 4 } } },
    id: undefined
  },
  Document {
    pageContent: "// Call the function\nhelloWorld();",
    metadata: { loc: { lines: { from: 6, to: 7 } } },
    id: undefined
  }
]
